In [ ]:
import torch
import triton
import triton.language as tl
from torch.library import triton_op, wrap_triton


def swiglu_ref(a, b):
    return torch.nn.functional.silu(a) * b

@triton.autotune(
    [
        triton.Config({"BLOCK_SIZE": BLOCK_SIZE, "VEC_SIZE": vec_size})
        for BLOCK_SIZE in (128, 256, 512, 1024)
        for vec_size in [1, 2, 4]
        if BLOCK_SIZE % vec_size == 0
    ],
    key=["n_elements"]
)
@triton.jit
def _swiglu_kernel_autotune(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr, VEC_SIZE: tl.constexpr, DTYPE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * (BLOCK_SIZE // VEC_SIZE) + tl.arange(0, BLOCK_SIZE // VEC_SIZE)
    mask = offsets < n_elements
    if DTYPE == tl.bfloat16:
        a = tl.load(x_ptr + offsets * VEC_SIZE, mask=mask).to(tl.float32)
    else:
        a = tl.load(x_ptr + offsets * VEC_SIZE, mask=mask)
    b = tl.load(y_ptr + offsets * VEC_SIZE, mask=mask)
    silu_a = a * tl.sigmoid(a)

    if DTYPE == tl.bfloat16:
        silu_a = silu_a.to(tl.bfloat16)

    result = silu_a * b

    tl.store(output_ptr + offsets * VEC_SIZE, result, mask=mask)

@triton_op("llm_scaling_week::swiglu_fwd", mutates_args={})
def swiglu_fwd(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    out = torch.empty_like(a)
    grid = lambda meta: (triton.cdiv(a.numel(), meta["BLOCK_SIZE"]),)

    dtype = a.dtype
    if dtype == torch.float32:
        _swiglu_kernel_autotune[grid](a, b, out, a.numel(), DTYPE=tl.float32)
    else:
        _swiglu_kernel_autotune[grid](a, b, out, a.numel(), DTYPE=tl.bfloat16)
    return out
